# zameen on Colab: run the repository

This runs the same pipeline as `zameen run` on a workstation, from a Colab A100. Results and the
trained adapter go to Google Drive, so nothing is lost when the session ends, and a re-run after a
disconnect resumes where it stopped.

**Runtime → Change runtime type → A100 (High-RAM) → Run all.** For Sarvam-30B start from a fresh
runtime (Runtime → Disconnect and delete runtime): it needs transformers 4.x.

In [ ]:
CONFIG = "qwen2.5-14b-medium"   # @param ["qwen2.5-14b-small", "qwen2.5-14b-medium", "sarvam-30b-small", "sarvam-30b-medium", "sarvam-30b-small-r64", "qwen2.5-7b-small"]
REPO_URL = "https://github.com/ukamathAppliedML/zameen.git"   # @param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/zameen_runs"             # @param {type:"string"}

In [ ]:
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
os.environ["HF_HOME"] = "/content/hf_cache"   # local disk: Sarvam's checkpoint is ~129 GB
if not os.path.isdir("/content/zameen"):
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/zameen"], check=True)
extra = "sarvam" if CONFIG.startswith("sarvam") else "train"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"/content/zameen[{extra}]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)   # conflicts with peft
!zameen doctor --model {CONFIG.rsplit('-', 1)[0] if not CONFIG.endswith('r64') else 'sarvam-30b'}

In [ ]:
%cd /content/zameen
!zameen run -c configs/{CONFIG}.yaml --output-dir {OUTPUT_DIR}

In [ ]:
from IPython.display import Image, Markdown, display
run_dir = f"{OUTPUT_DIR}/{CONFIG}"
display(Markdown(open(f"{run_dir}/results.md", encoding="utf-8").read()))
for fig in ("learning_curve.png", "before_after.png"):
    display(Image(f"{run_dir}/{fig}"))
print("Adapter:", f"{run_dir}/adapter  → copy into models/{CONFIG}/adapter/ in the repository")